<a href="https://colab.research.google.com/github/hanamonem/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanamonem/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions
### Finding 1
### Finding 1

The research paper reports that content refresh activity is associated with improved search performance.

My methodology question is: how was the outcome defined and measured? I would like to understand whether improvement was evaluated using future performance windows and whether external factors such as seasonality or search engine changes were considered.

### Finding 2

The paper suggests that trend-based signals can help identify refresh opportunities.

My methodology question is whether the validation design supports the claim. Specifically, I would like to know whether grouped or time-aware validation was used and whether results were evaluated on unseen content.

These questions are intended to strengthen confidence in the findings and improve interpretation of the results.

In [22]:
print("Rows:", len(df))

print("\nUnique clients:")
print(df["client_id"].nunique())

print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts(normalize=True))

Rows: 30000

Unique clients:
32

Trend direction distribution:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


## 2. My model under an honest split (before/after)
The original workflow used a standard random split.

To make the validation more realistic, I also evaluated the data using a grouped split based on client_id.

The grouped split prevents content from the same client from appearing in both training and testing data.

This reduces memorization risk and provides a more honest estimate of expected performance on unseen clients.

The difference between the random and grouped splits is itself a useful validation finding.

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit

url = "https://raw.githubusercontent.com/hanamonem/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Random split

train_random, test_random = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("RANDOM SPLIT")
print("Train rows:", len(train_random))
print("Test rows:", len(test_random))

# Grouped split

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=df["client_id"])
)

train_group = df.iloc[train_idx]
test_group = df.iloc[test_idx]

print("\nGROUPED SPLIT")
print("Train rows:", len(train_group))
print("Test rows:", len(test_group))

print("\nUnique train clients:",
      train_group["client_id"].nunique())

print("Unique test clients:",
      test_group["client_id"].nunique())

print("\nBase rate:")
print(df["trend_direction"].value_counts(normalize=True))

RANDOM SPLIT
Train rows: 24000
Test rows: 6000

GROUPED SPLIT
Train rows: 23837
Test rows: 6163

Unique train clients: 25
Unique test clients: 7

Base rate:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


## 3. Leakage audit
I reviewed the final feature set for possible leakage.

The fields trend_direction and trend_pct are potential leakage risks because they summarize content performance trends.

The columns client_id and content_id are identifiers and should not be used as predictive features.

No product flags, existing-system scores, or client-identifying information were used as model inputs.

Feature importance and feature definitions were reviewed to ensure that inputs remain suitable for decision-support purposes.

In [24]:
suspects = [
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
]

print("Potential leakage fields:\n")

for s in suspects:
    print("-", s)

Potential leakage fields:

- trend_direction
- trend_pct
- client_id
- content_id


## 4. Claim rewrite

Original claim:

The model identifies which pages should be refreshed and predicts future performance improvements.

Rewritten claim:

The analysis produces a directional ranking of content refresh opportunities based on observed search and engagement signals.

The output is intended for decision-support and prioritization rather than as a guarantee of future performance outcomes.

In [25]:
print("Claim rewritten using safe language.")


Claim rewritten using safe language.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.